In [1]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Dataset Paths
dataset_path = "D:/Dataset-lung"
batch_size = 32
image_size = (224, 224)  # Resized dimensions

# Preprocessing and Augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,               # Normalize pixel values to [0, 1]
    rotation_range=15,                 # Random rotation
    width_shift_range=0.1,             # Horizontal shift
    height_shift_range=0.1,            # Vertical shift
    shear_range=0.1,                   # Shear transformations
    zoom_range=0.1,                    # Random zoom
    horizontal_flip=True,              # Horizontal flipping
    fill_mode='nearest',               # Filling missing pixels
    validation_split=0.2               # Split data into training and validation
)

# Training Data Generator
train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',          # Multi-class classification
    subset='training',                 # Training subset
    shuffle=True
)

# Validation Data Generator
val_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',               # Validation subset
    shuffle=False
)

# Load ResNet50 without pre-trained weights
base_model = ResNet50(weights=None, include_top=False, input_shape=(224, 224, 3))

# Add custom classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)  # Fully connected layer
x = Dense(128, activation='relu')(x)  # Optional additional layer
predictions = Dense(4, activation='softmax')(x)  # Output layer for 4 classes

# Build the final model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Define Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint('resnet50_no_weights.keras', monitor='val_loss', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

# Train the model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks
)

# Save the final model
model.save('resnet50_no_weights.keras')

# Print class indices
print("Class Indices:", train_generator.class_indices)

Found 2856 images belonging to 4 classes.
Found 713 images belonging to 4 classes.
Epoch 1/20


C:\Users\BHANU\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


90/90 ━━━━━━━━━━━━━━━━━━━━ 1875s 16s/step - accuracy: 0.5844 - loss: 1.2637 - val_accuracy: 0.3773 - val_loss: 2.9170 - learning_rate: 0.0010
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 1327s 15s/step - accuracy: 0.7400 - loss: 0.7438 - val_accuracy: 0.3773 - val_loss: 9.1981 - learning_rate: 0.0010
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 1200s 13s/step - accuracy: 0.7603 - loss: 0.6511 - val_accuracy: 0.1950 - val_loss: 26.2442 - learning_rate: 0.0010
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 1122s 12s/step - accuracy: 0.7961 - loss: 0.5433 - val_accuracy: 0.4306 - val_loss: 2.7231 - learning_rate: 0.0010
Epoch 5/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 1103s 12s/step - accuracy: 0.8221 - loss: 0.4880 - val_accuracy: 0.4502 - val_loss: 1.9193 - learning_rate: 0.0010
Epoch 6/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 1094s 12s/step - accuracy: 0.7973 - loss: 0.5167 - val_accuracy: 0.1964 - val_loss: 4.2067 - learning_rate: 0.0010
Epoch 7/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 1103s 12s/step - accuracy: 0.8270 - loss: 0.4622 - val